# Péndulo invertido: solución teórica del sistema linealizado

*(Traducción a Python del livescript `pendulo_teorico.mlx`, usando `sympy`)*

Tomamos todos los parámetros del péndulo como 1, para simplificar.

In [1]:
import sympy as sp
sp.init_printing()

In [2]:
x1, x2, t = sp.symbols('x1 x2 t', real=True)

f = sp.Matrix([
    x2,
    sp.sin(x1) - x2
])
A = f.jacobian([x1, x2])
A

⎡   0     1 ⎤
⎢           ⎥
⎣cos(x₁)  -1⎦

**Ojo:** estamos linealizando en torno a $x_1=0$ (el péndulo "arriba", posición de equilibrio inestable), no en torno a $x_1=\pi$ (péndulo "colgando", equilibrio estable). Como $\frac{d}{dx_1}\sin(x_1) = \cos(x_1)$ y $\cos(0)=1$, obtenemos ese término positivo en el Jacobiano — si linealizásemos en $\pi$, saldría $f_2 = -\sin(x_1)-x_2$ y el signo cambiaría.

In [3]:
Anum = A.subs({x1: 0, x2: 1})
Anum

⎡0  1 ⎤
⎢     ⎥
⎣1  -1⎦

Obtenemos ahora la descomposición en autovalores/autovectores. `sympy` no tiene una función `eig` que devuelva `(autovectores, autovalores)` como el `[V,D]=eig(A)` de MATLAB; el equivalente es `.diagonalize()`, que devuelve `(P, D)` con $A = PDP^{-1}$: $P$ son los autovectores (en columnas, como en MATLAB) y $D$ los autovalores en la diagonal.

In [4]:
P, D = Anum.diagonalize()
print('P (autovectores, columnas) =')
display(P)
print('D (autovalores en la diagonal) =')
D

P (autovectores, columnas) =


⎡1   √5  1   √5⎤
⎢─ + ──  ─ - ──⎥
⎢2   2   2   2 ⎥
⎢              ⎥
⎣  1       1   ⎦

D (autovalores en la diagonal) =


⎡  1   √5          ⎤
⎢- ─ + ──     0    ⎥
⎢  2   2           ⎥
⎢                  ⎥
⎢            √5   1⎥
⎢   0      - ── - ─⎥
⎣            2    2⎦

Ahora necesitamos $e^{Dt}$. Como $D$ es diagonal, su exponencial es simplemente la exponencial de cada elemento de la diagonal por separado — no hace falta ninguna función especial de exponencial matricial simbólica para este caso particular (a diferencia de `expm` en MATLAB, que sí es una función general para cualquier matriz).

In [5]:
eAt = sp.diag(*[sp.exp(D[i, i]*t) for i in range(D.shape[0])])
eAt

⎡   ⎛  1   √5⎞               ⎤
⎢ t⋅⎜- ─ + ──⎟               ⎥
⎢   ⎝  2   2 ⎠               ⎥
⎢ℯ                    0      ⎥
⎢                            ⎥
⎢                  ⎛  √5   1⎞⎥
⎢                t⋅⎜- ── - ─⎟⎥
⎢                  ⎝  2    2⎠⎥
⎣      0        ℯ            ⎦

Y calculamos la solución del sistema linealizado para una condición inicial $x_0=(0.1,\ 0)$:

$$x(t) = Pe^{Dt}P^{-1}x_0$$

In [6]:
x0 = sp.Matrix([sp.Rational(1, 10), 0])
x = P * eAt * P.inv() * x0
x = sp.simplify(x)
x

⎡                           -t⋅(1 + √5)   ⎤
⎢                           ────────────  ⎥
⎢ ⎛    √5⋅t      √5⋅t    ⎞       2        ⎥
⎢ ⎝√5⋅ℯ     + 3⋅ℯ     + 2⎠⋅ℯ              ⎥
⎢ ──────────────────────────────────────  ⎥
⎢              10⋅(√5 + 5)                ⎥
⎢                                         ⎥
⎢                             -t⋅(1 + √5) ⎥
⎢                             ────────────⎥
⎢⎛ √5⋅t       √5⋅t         ⎞       2      ⎥
⎢⎝ℯ     + √5⋅ℯ     - √5 - 1⎠⋅ℯ            ⎥
⎢─────────────────────────────────────────⎥
⎣               10⋅(√5 + 5)               ⎦

Podemos evaluar esta expresión simbólica en distintos instantes de tiempo para observar el comportamiento,

In [7]:
for tv in [0, 1, 2, 5]:
    val = x.subs(t, tv).evalf()
    print(f't={tv}: x1={val[0]:.4f}, x2={val[1]:.4f}')

t=0: x1=0.1000, x2=0.0000
t=1: x1=0.1397, x2=0.0741
t=2: x1=0.2502, x2=0.1522
t=5: x1=1.5906, x2=0.9830


El sistema diverge. En realidad, solo se parece al péndulo cuando estás cerca de $\pi$, con $\delta x \approx 0$. Después el sistema linealizado se aleja indefinidamente del origen, ya que tienes una fuerza proporcional a $x_1$ que crece indefinidamente.

Sin embargo, el sistema no linealizado debe converger al origen, que es punto de equilibrio del péndulo invertido (en $x_1=0$ significa péndulo hacia abajo — recordad la nota de arriba sobre en qué punto estamos linealizando). Cuanto más te alejas en la condición inicial de $\pi$, más rápido divergen los resultados de ambos sistemas.